# 视图副本与内存布局

学习目标：判断数组操作是否共享数据，为独立修改或形状转换选择合适的复制方式，并检查连续性与只读边界。

前置知识：数组索引与切片、dtype、reshape、转置、Python 变量赋值。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的普通数值数组，后续单元沿用首次导入的 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 保留原始读数

修订一段测量数据时，如果原始记录需要保留，应先复制待修订的区域。基本切片会共享原数组的数据，直接修改切片中的元素也会改变原记录。

下面 readings 保存四次观测，draft 是中间两次观测的独立副本。copy() 复制数值数据，np.shares_memory() 检查两个数组是否有重叠的元素内存。

In [1]:
import numpy as np

readings = np.array([18, 20, 21, 23], dtype=np.int16)
draft = readings[1:3].copy()
draft[0] = 99

print(draft, draft.shape, draft.dtype)  # [99 21] (2,) int16。
print(readings)  # [18 20 21 23]，原始记录不变。
print(np.shares_memory(readings, draft))  # False，两个数组的数据独立。

[99 21] (2,) int16
[18 20 21 23]
False


## 2 别名、视图与副本

变量赋值可以让两个名称指向同一个数组，这里称为别名。view() 则创建新的数组对象，共享原来的数据；copy() 创建独立的数据副本。

数组既包含数值数据，也包含 shape、dtype 等描述信息。视图（view）可以有自己的形状，但访问的仍是共享数据。下面 view() 不传 dtype 参数，元素类型保持不变。

is 判断是否为同一个对象；shares_memory() 判断是否共享元素内存，两者回答的问题不同。

In [2]:
source = np.array([10, 20, 30], dtype=np.int16)
alias = source
view = source.view()
copied = source.copy()

print(alias is source, view is source, copied is source)  # True False False。
print(np.shares_memory(source, view))  # True，不同对象仍可共享数据。
print(np.shares_memory(source, copied))  # False。

alias[0] = 11
view[1] = 22
copied[2] = 99
print(source)  # [11 22 30]，别名和视图的修改影响原数组。
print(copied)  # [10 20 99]，副本独立修改。
print(view.shape, view.dtype)  # (3,) int16。

True False False
True
False
[11 22 30]
[10 20 99]
(3,) int16


## 3 索引读取与赋值

### 3.1 基本切片与高级索引读取

基本切片返回视图；整数数组或布尔数组等高级索引读取返回副本。即使两种写法选中了相同位置，共享关系也可能不同。

下面用 1:3 与 [1, 2] 选中同样的两个值，再分别修改读取结果。

In [3]:
source = np.array([10, 20, 30, 40], dtype=np.int16)
sliced = source[1:3]
selected = source[[1, 2]]

print(sliced, selected)  # 两者开始时都是 [20 30]。
print(sliced.shape, selected.shape, selected.dtype)  # (2,) (2,) int16。
print(np.shares_memory(source, sliced))  # True。
print(np.shares_memory(source, selected))  # False。

sliced[0] = 99
selected[1] = 88
print(source)  # [10 99 30 40]，只有切片修改写回原数组。
print(selected)  # [20 88]，高级索引读取的副本独立修改。

[20 30] [20 30]
(2,) (2,) int16
True
False
[10 99 30 40]
[20 88]


### 3.2 直接索引赋值

高级索引放在等号左侧时，会修改原数组的相应位置。“读取返回副本”不能理解成“高级索引永远不能修改原数组”。

下面先保存一次读取结果，再对原数组直接索引赋值。

In [4]:
source = np.array([10, 20, 30, 40], dtype=np.int16)
selected = source[[1, 2]]
source[[1, 2]] = [70, 80]

print(source)  # [10 70 80 40]，直接赋值修改原数组。
print(selected)  # [20 30]，之前取得的副本不随之改变。
print(source.shape, source.dtype)  # (4,) int16。

[10 70 80 40]
[20 30]
(4,) int16


## 4 base 与共享关系

base 指向数组所依赖的基础对象；自己拥有数据的数组，其 base 通常为 None。它描述数据来源，不能单独判断任意两个数组的元素是否重叠。

下面两个切片来自同一个基础数组，但分别选择偶数索引和奇数索引，实际没有共享的元素位置。需要判断两数组之间的重叠时，直接检查 shares_memory()。

In [5]:
source = np.array([10, 11, 12, 13, 14, 15], dtype=np.int16)
even_positions = source[::2]
odd_positions = source[1::2]

print(even_positions, odd_positions)  # [10 12 14] 与 [11 13 15]。
print(even_positions.base is odd_positions.base)  # True，基础对象相同。
print(np.shares_memory(even_positions, odd_positions))  # False，元素不重叠。
print(np.shares_memory(source, even_positions))  # True，与原数组有重叠。

even_positions[0] = 99
print(odd_positions)  # [11 13 15]，另一组位置没有改变。

[10 12 14] [11 13 15]
True
False
True
[11 13 15]


## 5 C 连续与 F 连续

连续性描述元素在内存中的排列，不由打印出来的表格外观决定。二维数组中，C 连续对应逐行排列，F 连续对应逐列排列；多维情况下分别是最后一个轴、第一轴变化最快。

| 属性 | 中文名称／含义 |
| --- | --- |
| flags.c_contiguous | 是否按 C 顺序连续存放 |
| flags.f_contiguous | 是否按 Fortran 顺序连续存放 |

下面 table 是两次观测、四个传感器组成的二维数组。转置交换轴但共享数据；间隔取列可能既不是 C 连续，也不是 F 连续。两个连续性标志并不互斥，例如连续一维数组可以同时满足两者。

In [6]:
table = np.array([[10, 11, 12, 13], [20, 21, 22, 23]], dtype=np.int16)
transposed = table.T
stepped = table[:, ::2]
vector = np.array([10, 20, 30], dtype=np.int16)

print(table.shape, table.flags.c_contiguous, table.flags.f_contiguous)
# (2, 4) True False，原数组逐行连续。
print(transposed.shape, transposed.flags.c_contiguous, transposed.flags.f_contiguous)
# (4, 2) False True，转置后按列连续。
print(stepped.shape, stepped.flags.c_contiguous, stepped.flags.f_contiguous)
# (2, 2) False False，列间有间隔。
print(vector.flags.c_contiguous, vector.flags.f_contiguous)  # True True。
print(np.shares_memory(table, transposed))  # True，转置没有复制数值数据。

(2, 4) True False
(4, 2) False True
(2, 2) False False
True True
True


## 6 reshape 的复制条件

### 6.1 能共享时返回视图

reshape() 在能够共享原数据时返回视图，否则需要复制。是否复制取决于现有布局、目标形状和读取顺序，不能只凭“是否连续”判断。

下面 table 的三行表示观测，四列表示传感器。每隔一列选取后，stepped 不是连续数组，但其元素仍能按固定间隔排成一维，因此这次 reshape(-1) 可以共享数据。-1 表示由元素数量推断一维长度。

In [7]:
table = np.arange(12, dtype=np.int16).reshape(3, 4)
stepped = table[:, ::2]
flat_view = np.reshape(stepped, -1)

print(stepped)  # 三行分别为 [0 2]、[4 6]、[8 10]。
print(stepped.flags.c_contiguous, stepped.flags.f_contiguous)  # False False。
print(flat_view, flat_view.shape, flat_view.dtype)  # [0 2 4 6 8 10] (6,) int16。
print(np.shares_memory(stepped, flat_view))  # True，本例不需要复制。

flat_view[0] = 99
print(table[0])  # [99 1 2 3]，通过一维视图修改原数组。

[[ 0  2]
 [ 4  6]
 [ 8 10]]
False False
[ 0  2  4  6  8 10] (6,) int16
True
[99  1  2  3]


### 6.2 读取顺序与禁止复制

reshape() 默认 order="C"，按最后一个轴变化最快的顺序读取和重排；order="F" 按第一个轴变化最快的顺序。这里的 order 指索引顺序，不是要求直接照搬输入的物理布局。

copy=None 是默认行为，需要时复制；copy=True 要求复制；copy=False 禁止复制，无法做到时触发 ValueError。下面同一个转置数组按 C 顺序展平需要复制，按 F 顺序则可以共享。

In [8]:
table = np.array([[10, 11, 12], [20, 21, 22]], dtype=np.int16)
transposed = table.T
flat_c = np.reshape(transposed, -1)
flat_f = np.reshape(transposed, -1, order="F", copy=False)

print(flat_c)  # [10 20 11 21 12 22]，逐行读取转置后的表格。
print(flat_f)  # [10 11 12 20 21 22]，逐列读取转置后的表格。
print(np.shares_memory(transposed, flat_c))  # False，本次必须重排数据。
print(np.shares_memory(transposed, flat_f))  # True。

# 预期 ValueError：本次按 C 顺序展平需要复制数据，copy=False 禁止复制。
np.reshape(transposed, -1, copy=False)

[10 20 11 21 12 22]
[10 11 12 20 21 22]
False
True


ValueError: Unable to avoid creating a copy while reshaping.

## 7 ravel 与 flatten

### 7.1 展平后的共享关系

ravel() 返回连续的一维数组，需要时才复制；flatten() 总是复制。需要独立修改一维结果时，flatten() 的意图更明确。

下面输入是 C 连续的二维数组，两个方法默认按 C 顺序展平，数值相同，但共享关系不同。

In [9]:
table = np.array([[10, 11, 12], [20, 21, 22]], dtype=np.int16)
raveled = table.ravel()
flattened = table.flatten()

print(raveled, flattened)  # 两者都是 [10 11 12 20 21 22]。
print(raveled.shape, flattened.shape, flattened.dtype)  # (6,) (6,) int16。
print(np.shares_memory(table, raveled))  # True，ravel 本次不需要复制。
print(np.shares_memory(table, flattened))  # False，flatten 总是复制。

raveled[0] = 99
flattened[1] = 88
print(table[0])  # [99 11 12]，flattened 的修改没有写回。

[10 11 12 20 21 22] [10 11 12 20 21 22]
(6,) (6,) int16
True
False
[99 11 12]


### 7.2 ravel 需要连续输出

reshape(-1) 可以返回带间隔的一维视图，ravel() 则要求一维结果连续。因此，两者即使按相同顺序得到相同数值，也可能有不同的复制行为。

下面重新构造三个观测、四个传感器的数据。间隔取列得到的六个元素不能直接作为连续的一维数据，ravel() 需要复制。

下图与代码一样使用 int16、每元素 2 字节示意间隔取列后的布局；偏移均相对于各自缓冲区起点。reshape 保留间隔视图，ravel 另建连续副本。

![下图与代码一样使用 int16、每元素 2 字节示意间隔取列后的布局；偏移均相对于各自缓冲区起点。reshape 保留间隔视图，ravel 另建连续副本。](image/06-strided-view-ravel.png)

In [10]:
table = np.arange(12, dtype=np.int16).reshape(3, 4)
stepped = table[:, ::2]
reshaped = stepped.reshape(-1)
raveled = stepped.ravel()

print(reshaped, raveled)  # 两者都是 [0 2 4 6 8 10]。
print(np.shares_memory(stepped, reshaped))  # True，允许元素之间留有间隔。
print(np.shares_memory(stepped, raveled))  # False，连续输出需要复制。
print(reshaped.flags.c_contiguous, raveled.flags.c_contiguous)  # False True。

[ 0  2  4  6  8 10] [ 0  2  4  6  8 10]
True
False
False True


## 8 只读标志

flags.writeable=False 可以禁止通过该数组写入数据，但它不是数据快照，也不会追溯锁定之前已创建的可写视图。

下面先创建可写视图，再把原数组设为只读。对原数组直接赋值会失败，已有视图却仍能改变共享数据。是否可写与是否共享，需要分别检查。

In [11]:
source = np.array([10, 20, 30], dtype=np.int16)
existing_view = source.view()
source.flags.writeable = False

print(source.flags.writeable, existing_view.flags.writeable)  # False True。

# 预期 ValueError：source 已设为只读，不能向它直接赋值。
source[0] = 99

False True


ValueError: assignment destination is read-only

In [12]:
existing_view[0] = 88
print(source)  # [88 20 30]，之前创建的可写视图仍能修改共享数据。
print(np.shares_memory(source, existing_view))  # True。

[88 20 30]
True


## 9 小视图保留原缓冲区

视图只显示一小段数据，也可能让整块原数据继续留在内存中。删除原变量名不会删除仍被视图引用的数据。

nbytes 只计算当前数组元素的字节数，不能用小视图的 nbytes 推断它让多少原始数据继续存活。如果只需要独立保留小片段，可以复制片段，再解除对视图的引用。这里用十个整数演示关系，不分配大型数组。

In [13]:
source = np.arange(10, dtype=np.int16)
small_view = source[2:4]

print(small_view.nbytes, source.nbytes)  # 4 20，小视图只有两个元素。
print(small_view.base is source)  # True，本例仍引用原数组。
del source
print(small_view)  # [2 3]，删除原名称后仍可访问数据。
print(small_view.base.nbytes)  # 20，原数组仍由该视图保留。

saved = small_view.copy()
del small_view
print(saved, saved.nbytes, saved.base is None)  # [2 3] 4 True，独立小副本。

4 20
True
[2 3]
20
[2 3] 4 True


## 10 选学：步幅与保守检查

### 10.1 strides 与字节偏移

strides 称为步幅，表示沿每个轴移动一个位置时需要跨过的字节数。它说明索引如何对应到数据位置。

二维数组 table 中，i 是行索引、j 是列索引，均取合法的非负值。元素相对 table 起始位置的字节偏移为 i × table.strides[0] + j × table.strides[1]。下面只读取步幅，不修改它。

形状 (2, 3) 的 C 连续 int16 数组每元素占 2 字节，下一行相同列相距 6 字节，下一列相距 2 字节。

In [14]:
table = np.array([[10, 11, 12], [20, 21, 22]], dtype=np.int16)
row_index, column_index = 1, 2
offset = row_index * table.strides[0] + column_index * table.strides[1]

print(table.strides, table.itemsize)  # (6, 2) 2，单位都是字节。
print(offset, table[row_index, column_index])  # 偏移 10 字节，对应值 22。
print(table.T.strides)  # (2, 6)，转置交换了两个轴的步幅。
print(table[:, ::-1].strides)  # (6, -2)，反向取列沿该轴向较低地址移动。

(6, 2) 2
10 22
(2, 6)
(6, -2)


### 10.2 may_share_memory 的保守判断

may_share_memory() 默认只检查两数组所涉及的内存边界。返回 True 表示可能共享，不能证明存在重叠元素；shares_memory() 默认做精确判断。

下面两个切片的位置交错，内存范围重叠，但元素本身不重叠。复杂步幅输入的精确检查可能很耗时，本章只检查小型普通数组。

In [15]:
source = np.array([10, 11, 12, 13, 14, 15], dtype=np.int16)
even_positions = source[::2]
odd_positions = source[1::2]

print(np.may_share_memory(even_positions, odd_positions))  # True，仅表示可能。
print(np.shares_memory(even_positions, odd_positions))  # False，实际元素不重叠。

True
False


## 本章小结

（1）别名是同一对象的另一名称；视图是共享数据的新对象；数值副本拥有独立数据。

（2）基本切片读取共享，高级索引读取复制；直接索引赋值修改原数组。

（3）reshape 和 ravel 是否复制需要结合布局与顺序判断，flatten 总是复制；base 不能替代两数组之间的共享检查。

（4）连续性、可写性和共享关系分别描述不同条件。小视图还可能保留较大的原缓冲区，独立保留片段时应考虑复制。

## 练习

（1）先预测原数组和三个变量的输出，再运行核对。分别说明别名、切片与副本如何影响原数组；补充 shares_memory() 检查。

In [16]:
source = np.array([1, 2, 3, 4], dtype=np.int16)
alias = source
region = source[1:3]
copied = source[1:3].copy()

# 先记录预测，再运行核对；不要仅根据变量名判断共享关系。
alias[0] = 10
region[0] = 20
copied[1] = 30
print(source)
print(region)
print(copied)
# 在此补充 shape、dtype 与共享关系的检查。

[10 20  3  4]
[20  3]
[ 2 30]


（2）将下面二维数组展平成一维，并把结果第一个值改为 99。要求原数组不变，在 reshape(-1)、ravel()、flatten() 中选择一种写法并说明理由。如果允许修改原数组，选择是否会变化？

In [17]:
table = np.array([[10, 11, 12], [20, 21, 22]], dtype=np.int16)

# 在此说明选择理由，创建并修改一维结果。
# 检查：结果 shape 为 (6,)，dtype 为 int16，首值为 99。
# 原数组左上角仍为 10；shares_memory() 应为 False。

（3）对下面间隔取列得到的数组，分别执行 reshape(-1) 和 ravel()。打印数值、连续性与共享关系，解释为什么两者可以数值相同而复制行为不同。

In [18]:
table = np.arange(12, dtype=np.int16).reshape(3, 4)
stepped = table[:, ::2]

# 在此创建两种一维结果并比较。
# 检查：结果均为 [0, 2, 4, 6, 8, 10]，shape 均为 (6,)。
# 结合“是否要求连续输出”解释共享检查结果，不能只说输入不连续。

（4）只需要长期保留下方数组的最后三个数值，之后可以丢弃其余数据。创建合适的结果并解释为什么不直接保留切片视图，再删除原变量名。如何确认结果独立，且不会通过它继续保留原数组？

In [19]:
source = np.arange(20, dtype=np.int16)

# 在此选择操作并说明理由；删除 source 前先检查共享关系。
# 检查：结果为 [17, 18, 19]，占 6 字节，base 为 None。
# 删除原名称后结果仍可读取；不能只凭视图的 nbytes 判断保留内存量。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | 别名与复制：[Quickstart — Copies and views](https://numpy.org/doc/2.5/user/quickstart.html#copies-and-views) 的 No copy at all、View or shallow copy、Deep copy；[view](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.view.html) 的定义与省略 dtype 条件；[copy](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.copy.html) 的独立数据与 object 浅复制边界。本章仅使用普通数值数组。索引：[Copies and views](https://numpy.org/doc/2.5/user/basics.copies.html) 的 Indexing operations；[Indexing on ndarrays](https://numpy.org/doc/2.5/user/basics.indexing.html) 的 Slicing and striding 中视图保留原数组的 Note、Advanced indexing 与 Assigning values。共享：[base](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.base.html) 的定义；[shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html) 的精确检查与 Warning；[may_share_memory](https://numpy.org/doc/2.5/reference/generated/numpy.may_share_memory.html) 的边界检查与 True 含义。形状与展平：[reshape](https://numpy.org/doc/2.5/reference/generated/numpy.reshape.html) 的 order、copy、Returns；[ravel](https://numpy.org/doc/2.5/reference/generated/numpy.ravel.html) 的连续输出、复制条件与 Notes；[flatten](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.flatten.html) 的复制约定。布局：[ndarray](https://numpy.org/doc/2.5/reference/arrays.ndarray.html#internal-memory-layout-of-an-ndarray) 的 Internal memory layout；[flags](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.flags.html) 的 C_CONTIGUOUS、F_CONTIGUOUS、WRITEABLE 与 Notes；[strides](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.strides.html) 的字节偏移和 Notes；[nbytes](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.nbytes.html) 的元素字节数。 |
| GitHub（NumPy 官方源码） | [arrayobject.c](https://github.com/numpy/numpy/blob/v2.5.0/numpy/_core/src/multiarray/arrayobject.c) 的 PyArray_FailUnlessWriteable：只读写入设置 ValueError。核查 v2.5.0 标签；用于补充 flags 文档中的只读行为，异常类型另与本章 NumPy 2.5.3 实际执行核对。 |
| Python 官方文档（Python 3.12） | [Simple statements — The del statement](https://docs.python.org/3.12/reference/simple_stmts.html#the-del-statement)：删除名称解除绑定；其他引用仍由数组视图规则决定。 |